In [9]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.vectorstores import Chroma

In [3]:
pdf_path = "../kb/CMT1_updated.pdf"
loader = PyPDFLoader(pdf_path)
pages = loader.load_and_split()
print(f"Loaded {len(pages)} pages from the PDF.")

Loaded 3 pages from the PDF.


In [4]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([page.page_content for page in pages])
print(f"Split {len(pages)} pages into {len(chunks)} chunks.")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {chunk}...")
    print("-------------------------------")

Split 3 pages into 9 chunks.
Chunk 1: page_content='Understanding Charcot-Marie Tooth Disease Type 1  
Charcot-Marie-Tooth Type 1 is just one of many different types of Charcot-Marie-Tooth disease. 
While the different CMT subtypes share similar symptoms, the underlying cause of each type of CMT 
varies. Understanding these causes is critically important for developing and administering 
treatments.  
CMT1 is a form of CMT that is inherited with autosomal dominance (its inheritance pattern). This 
means the disease occurs with at least one copy of the disease-causing gene, and affected 
individuals usually also have one normal copy of the gene on a pair of chromosomes that do not 
affect gender. In CMT1, the part of the nervous system that is dysfunctional is called myelin. Myelin is 
a wrapping around the parts of the nerves that facilitate rapid electrical signals to other parts of the 
body. When it is damaged, like in CMT1, that signaling can slow, weaken and become uncoordinated.'

In [7]:
embeddings = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"token": None}
)
embeddings

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2279.89it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={'token': None}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [8]:
embeddings_list = embeddings.embed_documents([chunk.page_content for chunk in chunks])
print(f"Generated embeddings for {len(chunks)} chunks.")
for i, embedding in enumerate(embeddings_list):
    print(f"Embedding for Chunk {i+1}: {embedding[:5]}...")  # Print the first 5 values of the embedding
    print("-------------------------------")

Generated embeddings for 9 chunks.
Embedding for Chunk 1: [-0.021968834102153778, -0.09972596168518066, 0.0809643343091011, 0.0738135352730751, -0.020228561013936996]...
-------------------------------
Embedding for Chunk 2: [-0.0561274029314518, -0.0931975468993187, 0.1300518810749054, 0.1039748266339302, -0.016551991924643517]...
-------------------------------
Embedding for Chunk 3: [-0.03783997520804405, -0.1217343807220459, 0.09287106245756149, 0.07922101020812988, 0.002534739440307021]...
-------------------------------
Embedding for Chunk 4: [-0.0944029912352562, -0.10340632498264313, 0.02805413119494915, 0.030978016555309296, 0.011204284615814686]...
-------------------------------
Embedding for Chunk 5: [-0.08075447380542755, -0.048039037734270096, 0.05354171618819237, 0.02983042411506176, 0.021698566153645515]...
-------------------------------
Embedding for Chunk 6: [-0.079897940158844, -0.06817552447319031, 0.08210297673940659, 0.040531206876039505, 0.0023370671551674604]..

In [11]:
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="chroma_db",
    collection_name="sample_collection"
)

In [12]:
similarity_search_results = vector_store.similarity_search_with_relevance_scores("What are symptoms of cmt2?", k=2)
print("Similarity Search Results:")
for result in similarity_search_results:
    print(result)

Similarity Search Results:
(Document(metadata={}, page_content='body. When it is damaged, like in CMT1, that signaling can slow, weaken and become uncoordinated. \nThe slowing of electrical signals and loss of coordinated signaling is what leads to muscle weakness.  \n  \n  \nSymptoms of CMT1  \nThe first symptom of CMT1 is usually muscle weakness in the feet. This presents as hammertoes, pes \ncavus (high arch) and other foot deformities, which are caused by the small muscles in the feet \ndeteriorating. Additionally, people with CMT1 often have a loss of sensation accompanying this \nweakness. Their symptoms affect both motor and sensory function because the myelin that is meant \nto insulate nerves is damaged, impacting communication with the musculature and the sensory \norgans in the skin. Charcot-Marie-Tooth Type 1 is a progressive disorder, meaning the symptoms will \nworsen over time, but this progression is usually slow. Eventually, the weakness and loss of sensation'), 0.4425